# Cost Estimates for Handling Injuries

A scenario estimate of workers' compensation and indirect costs for the lost-time handling injuries counted in [`injury-rates/`](../injury-rates/). Every input is either a count from this repository or a published unit cost, named with its source. The output is a range, not a measured cost.

**Inputs from this repository** `injury-rates/derived/lost-time.csv` (cases with at least one day away from work, OSHA ITA 2025) and the rule precision measured by blind review in `handling-task-types/`.

**Published unit costs**
- Low: $32,023, average direct cost of a strain claim in OSHA's $afety Pays estimator, from NCCI workers' compensation data (https://www.osha.gov/safetypays/estimator_text.html).
- High: $47,316, average cost of all lost-time workers' compensation claims for accidents in 2022–2023, NCCI, reported by the National Safety Council (https://injuryfacts.nsc.org/work/costs/workers-compensation-costs/).
- Indirect cost ratio: 1.1 times direct cost for claims of $10,000 or more, OSHA $afety Pays (https://www.osha.gov/safetypays/background).

**Not included** Cases with restricted duty but no days away; establishments with fewer than 100 employees; turnover, overtime beyond the indirect ratio, legal costs, and patient harm. The estimates are therefore low.

In [1]:
import os, pandas as pd
URL = 'https://raw.githubusercontent.com/yuxuan-huang-data/us-care-safety-signals/main/injury-rates/derived/lost-time.csv'
p = '../injury-rates/derived/lost-time.csv'
lt = pd.read_csv(p if os.path.exists(p) else URL, index_col=0)
UNIT = {'low': 32023, 'high': 47316}; INDIRECT = 1.1
PREC = {('hospital', 'handling'): 0.80, ('nursing facility', 'handling'): 0.893, ('hospital', 'lateral'): 0.833}
rows = []
for (setting, kind), prec in PREC.items():
    n = int(lt.loc[setting, f'{kind} lost-time cases']); adj = round(n * prec)
    for basis, cases in [('rule count', n), ('adjusted for precision', adj)]:
        r = {'setting': setting, 'injuries': 'lateral transfer' if kind == 'lateral' else 'handling', 'basis': basis, 'lost-time cases': cases,
             'days away': int(lt.loc[setting, f'{kind} days away']), 'days restricted': int(lt.loc[setting, f'{kind} days restricted'])}
        for k, u in UNIT.items():
            r[f'direct, {k} ($M)'] = round(cases * u / 1e6, 1)
            r[f'direct + indirect, {k} ($M)'] = round(cases * u * (1 + INDIRECT) / 1e6, 1)
        rows.append(r)
est = pd.DataFrame(rows)
os.makedirs('derived', exist_ok=True); est.to_csv('derived/cost-estimates.csv', index=False)
pd.DataFrame([{'input': 'unit cost, low', 'value': UNIT['low'], 'source': 'OSHA $afety Pays, strain, NCCI data'},
              {'input': 'unit cost, high', 'value': UNIT['high'], 'source': 'NCCI via NSC Injury Facts, all lost-time claims, AY 2022-2023'},
              {'input': 'indirect cost ratio', 'value': INDIRECT, 'source': 'OSHA $afety Pays, claims >= $10,000'}]).to_csv('derived/unit-costs.csv', index=False)
pd.set_option('display.width', 220); est

,setting,injuries,basis,lost-time cases,days away,days restricted,"direct, low ($M)","direct + indirect, low ($M)","direct, high ($M)","direct + indirect, high ($M)"
0,hospital,handling,rule count,8473,312566,461884,271.3,569.8,400.9,841.9
1,hospital,handling,adjusted for precision,6778,312566,461884,217.1,455.8,320.7,673.5
2,nursing facility,handling,rule count,2056,58178,125621,65.8,138.3,97.3,204.3
3,nursing facility,handling,adjusted for precision,1836,58178,125621,58.8,123.5,86.9,182.4
4,hospital,lateral transfer,rule count,743,28358,41597,23.8,50.0,35.2,73.8
5,hospital,lateral transfer,adjusted for precision,619,28358,41597,19.8,41.6,29.3,61.5
